<a href="https://colab.research.google.com/github/ilaydayda/NLP-Type-Analysis/blob/main/NLPtypeAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import torch
import snowballstemmer
from nltk.corpus import stopwords
from transformers import AutoTokenizer, AutoModel
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

nltk.download('stopwords')
nltk.download('punkt')
stop_words_tr = set(stopwords.words("turkish"))
stemmer = snowballstemmer.stemmer('turkish')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

spor_list = [
    "Futbol maçında hakem penaltı noktasını gösterdi.",
    "Şampiyonlar ligi finalinde galibiyet golü uzatmalarda geldi.",
    "Basketbol liginde smaç rekoru kırıldı, taraftarlar sahaya indi.",
    "Voleybol milli takımı Avrupa şampiyonasında altın madalya kazandı.",
    "Antrenman programında kondisyon ve taktik ağırlıklı çalışıldı."
]

moda_list = [
   "Bu sezonun trend renkleri pastel tonlar.",
        "Mankenler podyumda elbiseyi tanıttı.",
        "Kumaş kalitesi ipek ve saten karışımı.",
        "Kış kreasyonunda kabanlar dikkat çekiyor.",
        "Tasarımcı defilede alkış topladı.",
        "Ayakkabı çanta kombini çok şıktı.",
        "Vintage giyim tekrar moda oldu.",
        "Dikiş nakış detayları harikaydı.",
        "Defilede ünlü mankenler yürüdü."
]

sanat_list = [
    "Ressam tuvale yağlı boya ile çizdi.",
        "Heykeltıraş mermeri yontarak eser yaptı.",
        "Sergideki tablolar modern sanat eseriydi.",
        "Müze gezisinde antik heykelleri gördük.",
        "Tiyatro sahnesinde oyuncular devleşti.",
        "Klasik müzik konseri ruhumuzu dinlendirdi.",
        "Sanat galerisi yeni resimler sergiliyor.",
        "Şair şiirini duygulu bir şekilde okudu.",
        "Sinema filmi vizyona girdi."
]

raw_data = {
    "spor": spor_list * 5,  # Listeyi 5 kez tekrar eder
    "moda": moda_list * 5,
    "sanat": sanat_list * 5
}

data_list = []
for cat, texts in raw_data.items():
    for t in texts:
        data_list.append({"text": t, "label": cat.capitalize()})
df = pd.DataFrame(data_list)

def clean_text(text):
    text = text.lower() #
    text = re.sub(r'<.*?>|https?://\S+|www\.\S+|\d+|[^\w\s]', ' ', text) #
    words = nltk.word_tokenize(text) #
    filtered = [w for w in words if w not in stop_words_tr and len(w) > 2] #
    stems = stemmer.stemWords(filtered)
    return " ".join(stems)

df['cleaned_text'] = df['text'].apply(clean_text)


#  BoW  Naive Bayes
vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(df['cleaned_text']).toarray()
clf_bow = MultinomialNB().fit(X_bow, df['label'])

# TF-IDF  Naive Bayes
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_text']).toarray()
clf_tfidf = MultinomialNB().fit(X_tfidf, df['label'])

#  N-Gram
vectorizer_ngram = CountVectorizer(ngram_range=(2, 2))
X_ngram = vectorizer_ngram.fit_transform(df['cleaned_text']).toarray()
clf_ngram = MultinomialNB().fit(X_ngram, df['label'])

# Word2Vec + KNN (k=3)
tokenized_sentences = [simple_preprocess(sentence) for sentence in df['text']]
word2vec_model = Word2Vec(sentences=tokenized_sentences, vector_size=50, window=5, min_count=1, sg=0)

def get_w2v_vector(text):
    words = simple_preprocess(text)
    vectors = [word2vec_model.wv[w] for w in words if w in word2vec_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(50)

X_w2v = np.array([get_w2v_vector(t) for t in df['text']])
clf_w2v = KNeighborsClassifier(n_neighbors=3, metric='cosine').fit(X_w2v, df['label'])

# BERT + KNN (k=3)
model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

def get_bert_vector(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, 0, :].cpu().numpy().flatten()

X_bert = np.array([get_bert_vector(t) for t in df['text']])
clf_bert = LogisticRegression().fit(X_bert, df['label'])


def analiz_et(metin):
    c_text = clean_text(metin)
    print("\n" + "="*60)
    print(f"GİRİLEN METİN: {metin}")
    print(f"KÖKLER: {c_text.split()}")
    print("-" * 60)

    temsiller = {
        "BoW (NB)": (vectorizer_bow.transform([c_text]).toarray(), clf_bow),
        "TF-IDF (NB)": (tfidf_vectorizer.transform([c_text]).toarray(), clf_tfidf),
        "N-Gram (NB)": (vectorizer_ngram.transform([c_text]).toarray(), clf_ngram),
        "W2V (KNN)": (get_w2v_vector(metin).reshape(1, -1), clf_w2v),
        "BERT (loc)": (get_bert_vector(metin).reshape(1, -1), clf_bert)
    }

    print(f"{'Metot':<15} | {'Tahmin':<10} | {'Güven Skoru'}")
    print("-" * 48)
    for isim, (vec, m) in temsiller.items():
        tahmin = m.predict(vec)[0]
        skor = np.max(m.predict_proba(vec)) * 100
        print(f"{isim:<15} | {tahmin:<10} | %{skor:.2f}")

while True:
    u_input = input("\nMetin girin (Çıkış: q): ")
    if u_input.lower() == 'q': break
    if u_input.strip(): analiz_et(u_input)
